# Cycle Types in Young Double Cosets — companion notebook (draft v0.2)
Dean Kulik, QuHarmonics Research Group, ORCID 0009-0003-3128-8828.

Every numbered claim marked [VERIFIED] in the paper is re-run here for $n\le7$. Set `RUN_LARGE = True` to extend the checks to $n=8$ (about a minute per cell) and $n=9$ for the table-only computations (several minutes). The scripts `audit.py`, `spectrum.py`, `countM.py`, `best.py` and `sweep8.py` must sit in the same folder as this notebook.

In [1]:
RUN_LARGE = False
import itertools, math, time
from collections import defaultdict, Counter
from math import factorial, prod, comb
from fractions import Fraction
from audit import int_partitions, ctype, UF, run as orbit_audit
from spectrum import tables, Lam, rel_counts
from countM import TS, Mvals, ec
from best import best_ncycles
from sweep8 import burnside

ValueError: invalid literal for int() with base 10: '-f'

## Lemma 2.3 and Remark 8.2 — direct orbit counts, key invariance, inversion splitting, double-coset/table identity

In [2]:
for n in range(4, 8 if not RUN_LARGE else 8):
    orbit_audit(n)

NameError: name 'orbit_audit' is not defined

## Theorem 3.2 — non-containment census (Table 1)

In [3]:
def nsetparts(mu):
    c=Counter(mu); return factorial(sum(mu))//(prod(factorial(k) for k in mu)*prod(factorial(v) for v in c.values()))
for n in (5,6,7):
    perms=list(itertools.permutations(range(n))); lam=[ctype(p) for p in perms]; clsz=Counter(lam)
    cells=strict=ties=bad=0; gain=0.0
    for mu in int_partitions(n):
        if all(k==1 for k in mu): continue
        mult=nsetparts(mu); m=len(mu); b=[]
        for i,k in enumerate(mu): b+=[i]*k
        grp=defaultdict(Counter); Tof=[]
        for si,p in enumerate(perms):
            T=[[0]*m for _ in range(m)]
            for x in range(n): T[b[x]][b[p[x]]]+=1
            T=tuple(map(tuple,T)); Tof.append(T); grp[T][lam[si]]+=1
        Kf=prod(factorial(k) for k in mu)
        for si in range(len(perms)):
            T=Tof[si]; M=grp[T][lam[si]]; DC=Kf*Kf//prod(factorial(v) for r in T for v in r); Cl=clsz[lam[si]]
            cells+=mult; best=min(DC,Cl)
            if M<best: strict+=mult
            else:
                ties+=mult; bad+= mult if DC<=Cl else 0
            gain+=mult*(math.log2(best)-math.log2(M))
    print(f"n={n}: configurations {cells:,}  strict {strict:,}  ties {ties:,}  ties with |KsK|<=|C| {bad}  mean gain {gain/cells:.3f} bits")

NameError: name 'ctype' is not defined

## Theorem 4.2 / Example 4.5 — worked example

In [4]:
mu=(2,2); T=((1,1),(1,1))
print({l:int(v) for l,v in Mvals(T,mu).items()}, ' |KsK| =', prod(factorial(d) for d in mu)**2//prod(factorial(v) for r in T for v in r))

NameError: name 'Mvals' is not defined

## Theorems 4.2, 5.1, 6.1 — enumeration versus transition-system counting on every table

In [5]:
for n in range(5, 9 if RUN_LARGE else 8):
    t0=time.time(); perms=list(itertools.permutations(range(n))); lam=[ctype(p) for p in perms]
    tot=bad_rec=bad_best=bad_supp=0
    for mu in int_partitions(n):
        m=len(mu); b=[]
        for i,k in enumerate(mu): b+=[i]*k
        grp=defaultdict(Counter)
        for si,p in enumerate(perms):
            T=[[0]*m for _ in range(m)]
            for x in range(n): T[b[x]][b[p[x]]]+=1
            grp[tuple(map(tuple,T))][lam[si]]+=1
        for T,Cn in grp.items():
            tot+=1
            if {l:Fraction(v) for l,v in Cn.items()}!=Mvals(T,mu): bad_rec+=1
            if best_ncycles([list(r) for r in T],mu)!=Cn.get((n,),0): bad_best+=1
            if set(Cn)!=set(Lam(T)): bad_supp+=1
    print(f"n={n}: tables {tot}  recursion mismatches {bad_rec}  BEST mismatches {bad_best}  support mismatches {bad_supp}  [{time.time()-t0:.0f}s]")

NameError: name 'ctype' is not defined

## Theorem 5.3 — hook classes $(n-k,1^k)$

In [6]:
def hook_TS(T,k):
    m=len(T); tot=0
    for a in itertools.product(*[range(0,T[v][v]+1) for v in range(m)]):
        if sum(a)!=k: continue
        R=[list(r) for r in T]
        for v in range(m): R[v][v]-=a[v]
        tot+=prod(comb(T[v][v],a[v]) for v in range(m))*ec(R,m)
    return tot
tot=bad=0
for n in range(2,8):
    for mu in int_partitions(n):
        for T in tables(mu):
            MV=Mvals(T,mu); sc=Fraction(prod(factorial(d) for d in mu),prod(factorial(v) for r in T for v in r))
            for k in range(0,n-1):
                tot+=1; bad+= Fraction(hook_TS(T,k))*sc != MV.get(tuple([n-k]+[1]*k),0)
print(f"hook formula: {tot-bad}/{tot} exact")

NameError: name 'int_partitions' is not defined

## Table 2 — Burnside orbit counts and cells computed from tables alone

In [7]:
for n in range(1, 10 if RUN_LARGE else 8):
    t0=time.time(); b1,b2=burnside(n) if n<=8 or RUN_LARGE else (None,None)
    ntab,c1,c2=rel_counts(n)
    print(f"n={n}: tables {ntab}  |X/S_n| {b1}  |X/(S_n x C2)| {b2}  cells {c1}  cells+inv {c2}  [{time.time()-t0:.0f}s]")

NameError: name 'burnside' is not defined

## Measurement 9.1 — distinct values of $F$ from the recursion

In [8]:
def values(n):
    vals=set()
    for mu in int_partitions(n):
        m=len(mu); tabs=tables(mu); ix={t:i for i,t in enumerate(tabs)}; u=UF(len(tabs))
        for t,i in ix.items():
            for s in [a for a in range(m-1) if mu[a]==mu[a+1]]:
                p=list(range(m)); p[s],p[s+1]=s+1,s
                u.u(i, ix[tuple(tuple(t[p[a]][p[c]] for c in range(m)) for a in range(m))])
            u.u(i, ix[tuple(tuple(t[c][a] for c in range(m)) for a in range(m))])
        reps={}
        for t,i in ix.items(): reps.setdefault(u.f(i),t)
        for t in reps.values(): vals|={int(v) for v in Mvals(t,mu).values()}
    return sorted(vals)
for n in range(1, 10 if RUN_LARGE else 8):
    v=values(n); print(f"n={n}: {len(v)} values" + (f": {v}" if n==7 else ""))

NameError: name 'int_partitions' is not defined

## v0.2 — Lemma 2.3 exhaustive ($K\times K$ orbits = table classes, Fisher–Yates sizes)

In [9]:
import subprocess, sys
def sh(*a):
    r=subprocess.run([sys.executable,*a],capture_output=True,text=True); print(r.stdout or r.stderr)
sh('dcexh.py','3','4','5','6','7', *(['8'] if RUN_LARGE else []))

C:\Users\Developer\anaconda3\envs\nexus-ultimate\python.exe: can't open file 'D:\\@User Data\\Downloads\\dcexh.py': [Errno 2] No such file or directory



## v0.2 — Theorem 5.4 (dominant cycle), Theorem 5.6 (involutions)

In [11]:
sh('dominant.py')

C:\Users\Developer\anaconda3\envs\nexus-ultimate\python.exe: can't open file 'D:\\@User Data\\Downloads\\dominant.py': [Errno 2] No such file or directory



## v0.2 — Corollary 5.5 (near-hooks) and Theorem 6.1a (length-rooted recursion)

In [12]:
sh('nearhook.py')

C:\Users\Developer\anaconda3\envs\nexus-ultimate\python.exe: can't open file 'D:\\@User Data\\Downloads\\nearhook.py': [Errno 2] No such file or directory



## v0.2 — Measurement 10.1 (what the table carries)

In [13]:
sh('tsuff.py','4','5','6','7', *(['8'] if RUN_LARGE else []))

C:\Users\Developer\anaconda3\envs\nexus-ultimate\python.exe: can't open file 'D:\\@User Data\\Downloads\\tsuff.py': [Errno 2] No such file or directory

